In [ ]:
def load_eeg(Patient_number, task, base_path="/home/bxai2/CYJ/Project/NT/NeuroTalk/Patients/preprocessed/"):
    """
    피험자 eeg 데이터 로드, resting state 따로 저장

    args:
        patient_number: 피험자 번호(내가 지정) ex. "sub001"
        task: Spoken or Imagined(내가 지정)
    """

    # 피험자 데이터 경로 설정
    patient_path = os.path.join(base_path, Patient_number, task)
    if not os.path.exists(patient_path):
        print(f"{Patient_number}의 데이터 경로가 존재하지 않음")
        return None, None
    
    # mff파일(eeg) 찾기
    mff_file = None
    for component in os.listdir(patient_path):
        if component.endswith('.mff'):
            mff_file = component
            break
    
    if not mff_file:
        print(f"{Patient_number}의 mff 파일을 찾을 수 없음")
        return None, None
    
    # eeg 데이터 로드
    raw_data = mne.io.read_raw_egi(os.path.join(patient_path, mff_file), preload=True)
    data_get = raw_data.get_data()
    channel_names = raw_data.ch_names


    # resting state 저장 경로 설정
    Rest_state_path = os.path.join(patient_path, "Resting_State")
    subject_dir = f"sub{Patient_number[-3:]}"
    save_dir = os.path.join(Rest_state_path, subject_dir)

    os.makedirs(save_dir, exist_ok=True)

    # resting state 추출
    Rest_state_EEG = data_get[:, 10*250:70*250]
    save_path = os.path.join(save_dir, f"{Patient_number}_{task}_resting_state.csv")

    pd.DataFrame(Rest_state_EEG.T, columns=channel_names).to_csv(save_path, index=False)
    print(f"{Patient_number}의 Resting state eeg 데이터 저장 완료")

    return data_get, channel_names